# Phase 1 — Data Ingestion & Validation
MIG Cement Demand Forecasting

Before any EDA or modeling, this notebook:
1. Ingests the raw operational data from SQLite
2. Validates schema consistency
3. Handles missing values
4. Detects and fixes negative inventory/consumption entries
5. Validates the inventory flow balance equation across all records
6. Saves a clean, versioned dataset that every later notebook reads from

**Data sources** (per project data requirements):
- **Operational data**: `consumed_tonnes`, `planned_pour_tonnes`, `silo_capacity` — on `Operations`/`Sites`
- **Logistics data**: `opening_inventory_tonnes`, `deliveries_tonnes`, `closing_inventory_tonnes` — on `Operations`
- **External data**: `rain_mm`, `avg_temp_c` — already embedded on `Operations`, not a separate source to join

All three categories live in the same `Operations` table joined to `Sites` for
site attributes (`region`, `behavior`, `silo_capacity`) — one join covers
everything.

## 1. Data Ingestion

Connect to the SQLite database and join `Operations` with `Sites`.

In [1]:
import sqlite3
import pandas as pd
import numpy as np
from pathlib import Path

DB_PATH = Path("../data/raw/MIG_Cement_Records.db")
conn = sqlite3.connect(DB_PATH)

query = """
SELECT a.*, s.region, s.behavior
FROM Operations AS a
JOIN Sites AS s ON a.site_id = s.site_id
"""

df = pd.read_sql_query(query, conn)
conn.close()

df["date"] = pd.to_datetime(df["date"])
print(df.shape)
df.head()

(32880, 13)


,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,North,aggressive
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,North,aggressive
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,North,aggressive
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,North,aggressive
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,North,aggressive


Note on the join: `s.silo_capacity` is deliberately **not** pulled in a
second time — `Operations` already carries its own `silo_capacity` column,
and the two are verified consistent below. Selecting `s.*` (as in the
original query) would create duplicate `silo_capacity` and `site_id`
columns, which pandas silently disambiguates with `.1` suffixes — a common
source of "why do I have two silo_capacity columns" confusion later.

## 2. Schema consistency validation

Checks that the data matches what the rest of the pipeline assumes:
expected columns, correct types, valid keys, and no duplicate records at
the stated grain (one row per date × site × cement_type).

In [2]:
expected_columns = {
    "date", "site_id", "cement_type", "planned_pour_tonnes", "consumed_tonnes",
    "opening_inventory_tonnes", "deliveries_tonnes", "closing_inventory_tonnes",
    "rain_mm", "avg_temp_c", "silo_capacity", "region", "behavior",
}
missing_cols = expected_columns - set(df.columns)
extra_cols = set(df.columns) - expected_columns
assert not missing_cols, f"Missing expected columns: {missing_cols}"
print("All expected columns present.")
if extra_cols:
    print("Extra columns found (not necessarily a problem):", extra_cols)

print()
print("--- dtypes ---")
print(df.dtypes)

All expected columns present.

--- dtypes ---
date                        datetime64[us]
site_id                                str
cement_type                            str
planned_pour_tonnes                float64
consumed_tonnes                    float64
opening_inventory_tonnes           float64
deliveries_tonnes                  float64
closing_inventory_tonnes           float64
rain_mm                            float64
avg_temp_c                         float64
silo_capacity                        int64
region                                 str
behavior                               str
dtype: object


In [3]:
# Grain check: exactly one row per (date, site_id, cement_type)
dup_count = df.duplicated(subset=["date", "site_id", "cement_type"]).sum()
print(f"Duplicate (date, site_id, cement_type) rows: {dup_count}")
assert dup_count == 0, "Grain violated — investigate duplicates before proceeding."

# Referential integrity: every cement_type in Operations should exist in CementTypes
conn = sqlite3.connect(DB_PATH)
valid_types = pd.read_sql_query("SELECT cement_type FROM CementTypes", conn)["cement_type"].tolist()
conn.close()

bad_types = df.loc[~df["cement_type"].isin(valid_types), "cement_type"].unique()
print(f"Cement types in data: {df['cement_type'].unique().tolist()}")
print(f"Unrecognized cement types: {list(bad_types) if len(bad_types) else 'none'}")
assert len(bad_types) == 0

# Date coverage: every site should have a full, gapless daily series
date_range = pd.date_range(df["date"].min(), df["date"].max(), freq="D")
expected_rows_per_site = len(date_range) * df["cement_type"].nunique()
rows_per_site = df.groupby("site_id").size()
sites_with_gaps = rows_per_site[rows_per_site != expected_rows_per_site]
print(f"\nExpected rows per site: {expected_rows_per_site}")
print(f"Sites with incomplete coverage: {len(sites_with_gaps)}")
if len(sites_with_gaps):
    print(sites_with_gaps)

Duplicate (date, site_id, cement_type) rows: 0
Cement types in data: ['CEM_II', 'CEM_I', 'CEM_III']
Unrecognized cement types: none

Expected rows per site: 3288
Sites with incomplete coverage: 30
site_id
SITE_001    1096
SITE_002    1096
SITE_003    1096
SITE_004    1096
SITE_005    1096
SITE_006    1096
SITE_007    1096
SITE_008    1096
SITE_009    1096
SITE_010    1096
SITE_011    1096
SITE_012    1096
SITE_013    1096
SITE_014    1096
SITE_015    1096
SITE_016    1096
SITE_017    1096
SITE_018    1096
SITE_019    1096
SITE_020    1096
SITE_021    1096
SITE_022    1096
SITE_023    1096
SITE_024    1096
SITE_025    1096
SITE_026    1096
SITE_027    1096
SITE_028    1096
SITE_029    1096
SITE_030    1096
dtype: int64


## 3. Missing value handling

Check every column for nulls. This dataset was found to be fully complete
in the original Phase 1 audit, but this check is written to actually
**handle** missing values (not just report them), so it stays correct if
the underlying data is refreshed and gaps appear later.

Strategy if any are found:
- `consumed_tonnes`, `deliveries_tonnes`: missing means "no activity that
  day" → fill with 0, not the mean (a mean would fabricate false demand).
- `rain_mm`, `avg_temp_c`: missing means a weather reading was lost, not
  that weather didn't happen → forward-fill within site (same-region
  weather doesn't change instantly day to day), fallback to site median.
- `opening_inventory_tonnes` / `closing_inventory_tonnes`: these should
  never be filled with a guess — a missing inventory balance is a real
  data problem worth flagging to whoever owns the source system, not
  papering over silently.

In [4]:
null_counts = df.isnull().sum()
null_counts = null_counts[null_counts > 0]

if len(null_counts) == 0:
    print("No missing values found across any column. Nothing to fix.")
else:
    print("Missing values found:")
    print(null_counts)

    if "consumed_tonnes" in null_counts.index:
        df["consumed_tonnes"] = df["consumed_tonnes"].fillna(0)
    if "deliveries_tonnes" in null_counts.index:
        df["deliveries_tonnes"] = df["deliveries_tonnes"].fillna(0)

    for col in ["rain_mm", "avg_temp_c"]:
        if col in null_counts.index:
            df[col] = df.groupby("site_id")[col].transform(lambda s: s.ffill())
            df[col] = df[col].fillna(df.groupby("site_id")[col].transform("median"))

    for col in ["opening_inventory_tonnes", "closing_inventory_tonnes"]:
        if col in null_counts.index:
            flagged = df[df[col].isnull()][["date", "site_id", "cement_type"]]
            print(f"\n{len(flagged)} rows with missing {col} — flagged for manual review, NOT auto-filled:")
            print(flagged.head(10))

    print("\nRemaining nulls after handling:")
    print(df.isnull().sum()[df.isnull().sum() > 0])

No missing values found across any column. Nothing to fix.


## 4. Negative value detection & correction

Physically, none of `consumed_tonnes`, `planned_pour_tonnes`,
`opening_inventory_tonnes`, `deliveries_tonnes`, `closing_inventory_tonnes`,
or `rain_mm` should ever be negative. `avg_temp_c` legitimately can be
(UK winter frost).

Where negatives are found, they're clipped to 0 and counted — cement
demand data commonly picks up negative values from correction/reversal
entries in source ERP systems (e.g. a delivery logged, then reversed with
a negative row instead of deleting the original).

In [5]:
non_negative_cols = [
    "consumed_tonnes", "planned_pour_tonnes", "opening_inventory_tonnes",
    "deliveries_tonnes", "closing_inventory_tonnes", "rain_mm",
]

negative_report = {}
for col in non_negative_cols:
    n_negative = (df[col] < 0).sum()
    if n_negative > 0:
        negative_report[col] = {
            "count": int(n_negative),
            "min_value": float(df[col].min()),
        }
        df[col] = df[col].clip(lower=0)

if negative_report:
    print("Negative values found and clipped to 0:")
    for col, info in negative_report.items():
        print(f"  {col}: {info['count']} rows, min was {info['min_value']}")
else:
    print("No negative values found in any non-negative column. Nothing to fix.")

print(f"\navg_temp_c range (negatives expected/valid here): "
      f"{df['avg_temp_c'].min():.1f}°C to {df['avg_temp_c'].max():.1f}°C")

No negative values found in any non-negative column. Nothing to fix.

avg_temp_c range (negatives expected/valid here): -5.0°C to 35.0°C


## 4b. Physical constraint: consumed cannot exceed available supply

A subtler check than the balance equation itself: `consumed_tonnes`
should never exceed what was actually available that day
(`opening_inventory_tonnes + deliveries_tonnes`) — you cannot consume
material you don't have. If this is violated, it means the recorded
consumption value is itself a data error, and the balance-equation
correction in the next section would otherwise silently produce a
negative `closing_inventory_tonnes` to "balance" an impossible input.
Caught during Step 7 production-hardening (a pytest regression test on
adversarial synthetic data found this; the real MIG dataset never
triggers it in practice, since Phase 1 found 0 negative closing-inventory
rows — but the pipeline should be correct regardless of how clean future
production data turns out to be).

In [6]:
available = df["opening_inventory_tonnes"] + df["deliveries_tonnes"]
TOLERANCE_AVAILABLE = 0.01  # same rounding tolerance as the balance-equation check below
over_available = df["consumed_tonnes"] > available + TOLERANCE_AVAILABLE

print(f"Rows where consumed_tonnes exceeds available supply: {over_available.sum()} / {len(df)}")
if over_available.sum() > 0:
    df.loc[over_available, "consumed_tonnes"] = available[over_available]
    print(f"Capped consumed_tonnes at available supply for {over_available.sum()} rows.")
else:
    print("No rows affected — consumed_tonnes never exceeds available supply in this dataset.")

Rows where consumed_tonnes exceeds available supply: 0 / 32880
No rows affected — consumed_tonnes never exceeds available supply in this dataset.


## 5. Inventory flow balance equation

Every row must satisfy:

**closing_inventory = opening_inventory + deliveries − consumed**

This is the single most important structural check in the whole dataset —
if this doesn't hold, every downstream inventory/reorder-point calculation
in Phase 4 is built on sand. A small floating-point tolerance (0.01 tonnes)
is allowed for rounding; anything beyond that gets corrected by
recalculating `closing_inventory_tonnes` from the equation itself
(opening + deliveries + consumed are the actual recorded transactions;
closing is derived, so it's the one to trust least when there's a
mismatch). With the previous section's cap in place, `expected_closing`
here is now guaranteed non-negative.

In [7]:
df["expected_closing"] = (
    df["opening_inventory_tonnes"] + df["deliveries_tonnes"] - df["consumed_tonnes"]
)
df["balance_diff"] = (df["closing_inventory_tonnes"] - df["expected_closing"]).abs()

TOLERANCE = 0.01
violations = df[df["balance_diff"] > TOLERANCE]
print(f"Balance equation violations (tolerance={TOLERANCE}): {len(violations)} / {len(df)}")

if len(violations):
    print("\nSample violations:")
    print(violations[["date", "site_id", "cement_type", "opening_inventory_tonnes",
                       "deliveries_tonnes", "consumed_tonnes",
                       "closing_inventory_tonnes", "expected_closing", "balance_diff"]].head(10))

    # Correct closing_inventory_tonnes to match the equation for violating rows only
    df.loc[df["balance_diff"] > TOLERANCE, "closing_inventory_tonnes"] = \
        df.loc[df["balance_diff"] > TOLERANCE, "expected_closing"]
    print(f"\nCorrected {len(violations)} rows' closing_inventory_tonnes to match the equation.")
else:
    print("Balance equation holds across all records. No correction needed.")

df = df.drop(columns=["expected_closing", "balance_diff"])

Balance equation violations (tolerance=0.01): 2 / 32880

Sample violations:
            date   site_id cement_type  opening_inventory_tonnes  \
25208 2022-01-01  SITE_024      CEM_II                     73.23   
28496 2022-01-01  SITE_027      CEM_II                     62.79   

       deliveries_tonnes  consumed_tonnes  closing_inventory_tonnes  \
25208              36.61            64.29                     45.56   
28496              13.10            12.38                     63.52   

       expected_closing  balance_diff  
25208             45.55          0.01  
28496             63.51          0.01  

Corrected 2 rows' closing_inventory_tonnes to match the equation.


## 6. Silo capacity overflow check

Separate from the balance equation — this checks whether
`closing_inventory_tonnes` ever physically exceeds the site's
`silo_capacity`, which shouldn't be possible for a real silo.

If found, inventory is capped at capacity and the excess is captured in a
new `overflow_tonnes` column (a proxy for material write-off/waste) rather
than silently discarded — this preserves the information instead of
just hiding it.

In [8]:
df["overflow_tonnes"] = (df["closing_inventory_tonnes"] - df["silo_capacity"]).clip(lower=0)
n_over = (df["overflow_tonnes"] > 0).sum()
pct_over = n_over / len(df) * 100
print(f"Rows exceeding silo capacity: {n_over} ({pct_over:.1f}%)")

if n_over > 0:
    print("\nBy behavior:")
    print(df.groupby("behavior").apply(lambda g: (g["overflow_tonnes"] > 0).mean()).round(3))

    df["closing_inventory_raw_tonnes"] = df["closing_inventory_tonnes"]
    df["closing_inventory_tonnes"] = df[["closing_inventory_tonnes", "silo_capacity"]].min(axis=1)
    print("\nclosing_inventory_tonnes capped at silo_capacity.")
    print("Raw (uncapped) values preserved in closing_inventory_raw_tonnes for reference.")

Rows exceeding silo capacity: 11439 (34.8%)

By behavior:
behavior
aggressive      0.000
chaotic         0.222
conservative    0.987
dtype: float64

closing_inventory_tonnes capped at silo_capacity.
Raw (uncapped) values preserved in closing_inventory_raw_tonnes for reference.


## 7. Stockout risk flag

Not a correction, but a derived field the rest of the project needs:
whether a scheduled pour could actually be fully supplied that day. This
directly feeds the ≥98% pour-readiness target outcome.

In [9]:
df["stockout_risk"] = df["consumed_tonnes"] < df["planned_pour_tonnes"]
df["pour_shortfall_tonnes"] = (df["planned_pour_tonnes"] - df["consumed_tonnes"]).clip(lower=0)

print(f"Overall stockout-risk rate: {df['stockout_risk'].mean()*100:.1f}%")
print(df.groupby("behavior")["stockout_risk"].mean().round(3))

Overall stockout-risk rate: 39.7%
behavior
aggressive      0.650
chaotic         0.218
conservative    0.143
Name: stockout_risk, dtype: float64


## 8. Final validation summary & save

Re-run every check above on the corrected dataframe to confirm nothing
was missed, then save as the single clean dataset every later notebook
(EDA, feature engineering, modeling) reads from.

In [10]:
print("=== FINAL VALIDATION SUMMARY ===")
print(f"Rows: {len(df)}")
print(f"Nulls remaining: {df.isnull().sum().sum()}")
print(f"Negative values remaining: "
      f"{sum((df[c] < 0).sum() for c in non_negative_cols)}")

# Balance check must account for capacity capping: for capped rows,
# closing_inventory_tonnes = silo_capacity by construction, not the raw
# equation. So compare against min(expected, silo_capacity), not
# expected directly — otherwise every capped row falsely reports as a
# "violation" here even though step 6 handled it correctly.
raw_expected = df["opening_inventory_tonnes"] + df["deliveries_tonnes"] - df["consumed_tonnes"]
expected_after_cap = raw_expected.clip(upper=df["silo_capacity"])
final_diff = (df["closing_inventory_tonnes"] - expected_after_cap).abs()
print(f"Balance equation violations remaining (>0.01, cap-adjusted): {(final_diff > 0.01).sum()}")
print(f"Rows still exceeding silo capacity: {(df['closing_inventory_tonnes'] > df['silo_capacity']).sum()}")
print("=================================")

=== FINAL VALIDATION SUMMARY ===
Rows: 32880
Nulls remaining: 0
Negative values remaining: 0
Balance equation violations remaining (>0.01, cap-adjusted): 0
Rows still exceeding silo capacity: 0


In [11]:
OUT_PATH = Path("../data/processed/operations_clean.parquet")
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(OUT_PATH, index=False)
print(f"Saved clean dataset -> {OUT_PATH} ({len(df)} rows, {len(df.columns)} columns)")
df.head()

Saved clean dataset -> ../data/processed/operations_clean.parquet (32880 rows, 17 columns)

,date,site_id,cement_type,planned_pour_tonnes,consumed_tonnes,opening_inventory_tonnes,deliveries_tonnes,closing_inventory_tonnes,rain_mm,avg_temp_c,silo_capacity,region,behavior,overflow_tonnes,closing_inventory_raw_tonnes,stockout_risk,pour_shortfall_tonnes
0,2022-01-01,SITE_001,CEM_II,43.18,34.54,52.56,45.83,63.85,3.40,-3.10,448,North,aggressive,0.0,63.85,True,8.64
1,2022-01-02,SITE_001,CEM_I,45.26,45.26,63.85,19.97,38.56,3.23,14.28,448,North,aggressive,0.0,38.56,False,0.00
2,2022-01-03,SITE_001,CEM_III,38.69,38.69,38.56,47.19,47.06,2.64,6.40,448,North,aggressive,0.0,47.06,False,0.00
3,2022-01-04,SITE_001,CEM_I,33.16,33.16,47.06,18.74,32.64,8.25,14.23,448,North,aggressive,0.0,32.64,False,0.00
4,2022-01-05,SITE_001,CEM_III,56.88,47.04,32.64,14.40,0.00,2.69,8.97,448,North,aggressive,0.0,0.00,True,9.84
